In [ ]:
import pandas as pd
import numpy as np
import gc
import os
import subprocess
import sys
from pathlib import Path
import seaborn as sns
from itertools import combinations
from scipy.stats import norm
from itertools import product
import plotly.graph_objects as go

In [ ]:
df = pd.read_parquet('d.parquet')

In [ ]:
feature_cols = ['open', 'high', 'low', 'close', 'volume', 'turnover', 
       'buy_ratio', 'sell_ratio', 'long_short_ratio', 'ratio_delta',
       'premium_close', 'mark_close', 'index_close', 'mark_index_spread',
       'basis_bps', 'funding_rate']
df.index = pd.to_datetime(df["datetime"])

In [ ]:
def walk_forward_splits(
    index,
    train_years=2,
    val_months=4,
    test_months=1,
    step_months=1
):
    split_date = index.min() + pd.DateOffset(years=train_years)
    splits = []

    while True:
        train_start = split_date - pd.DateOffset(years=train_years)
        train_end = split_date
        val_end = train_end + pd.DateOffset(months=val_months)
        test_end = val_end + pd.DateOffset(months=test_months)

        if test_end > index.max():
            break

        train_idx = (index >= train_start) & (index < train_end)
        val_idx = (index >= train_end) & (index < val_end)
        test_idx = (index >= val_end) & (index < test_end)

        splits.append((train_idx, val_idx, test_idx))
        split_date += pd.DateOffset(months=step_months)

    return splits

In [ ]:
X = df[feature_cols]
y = df['target1h']
splits = walk_forward_splits(df.index)

In [ ]:
sampler_names = [
    "TPESampler",
    "GPSampler",
    "CmaEsSampler",
    "QMCSampler",
]
random_seeds = [1, 2, 10]
n_trials = 50
thread_count = 1
skip_existing = True

worker_path = Path("ctb_optuna_worker.py").resolve()
worker_env = os.environ.copy()
worker_env["PYTHONUNBUFFERED"] = "1"
worker_env["OMP_NUM_THREADS"] = str(thread_count)
worker_env["MKL_NUM_THREADS"] = str(thread_count)
worker_env["OPENBLAS_NUM_THREADS"] = str(thread_count)

for variable_name in ("X", "y", "splits"):
    globals().pop(variable_name, None)
gc.collect()

for sampler_name in sampler_names:
    for seed in random_seeds:
        run_name = f"{sampler_name}_{seed}"
        output_path = Path(f"{run_name}.parquet").resolve()

        if skip_existing and output_path.exists():
            print(f"SKIP {run_name}: parquet already exists", flush=True)
            continue

        print(f"\n=== {run_name} ===", flush=True)
        subprocess.run(
            [
                sys.executable,
                str(worker_path),
                "--sampler", sampler_name,
                "--seed", str(seed),
                "--data", str(Path("d.parquet").resolve()),
                "--output", str(output_path),
                "--n-trials", str(n_trials),
                "--threads", str(thread_count),
            ],
            cwd=worker_path.parent,
            env=worker_env,
            check=True,
        )
        gc.collect()

In [ ]:
models = {
    f"{sampler_name}_{seed}": pd.read_parquet(
        f"{sampler_name}_{seed}.parquet"
    )
    for sampler_name in sampler_names
    for seed in random_seeds
}

In [ ]:
def dm_test(df1, df2):
    data = (
        df1[["y_true", "y_pred"]]
        .rename(columns={"y_pred": "pred1"})
        .join(
            df2[["y_pred"]].rename(columns={"y_pred": "pred2"}),
            how="inner"
        )
        .dropna()
    )

    e1 = data["y_true"] - data["pred1"]
    e2 = data["y_true"] - data["pred2"]

    d = e1**2 - e2**2

    dm = d.mean() / np.sqrt(d.var(ddof=1) / len(d))
    p = 2 * (1 - norm.cdf(abs(dm)))

    return dm, p

results = []

for m1, m2 in combinations(models.keys(), 2):
    dm, p = dm_test(models[m1], models[m2])
    results.append([m1, m2, dm, p])

results = pd.DataFrame(
    results,
    columns=["Model1", "Model2", "DM", "p-value"]
)

results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap

models = sorted(set(results["Model1"]).union(results["Model2"]))

p_matrix = pd.DataFrame(np.nan, index=models, columns=models)
np.fill_diagonal(p_matrix.values, 0)

for _, row in results.iterrows():
    m1, m2, p = row["Model1"], row["Model2"], row["p-value"]
    p_matrix.loc[m1, m2] = p
    p_matrix.loc[m2, m1] = p

mask = (p_matrix > 0.05).astype(int)

plt.figure(figsize=(10, 8))
sns.heatmap(
    mask,
    cmap=ListedColormap(["royalblue", "red"]),
    cbar=False,
    linewidths=0.5,
    linecolor="black",
    square=True
)

plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('DM.png')
plt.show()

In [ ]:
def walk_forward_positions(
    df,
    long_grid=np.linspace(0.70, 0.98, 15),
    short_grid=np.linspace(0.02, 0.30, 15),
    objective="pnl",
    commission=0.001
):
    df = df.sort_index().copy()
    folds = sorted(df["fold"].unique())
    result = []

    log_commission = np.log1p(commission)

    for test_fold in folds[2:]:

        train = df[df["fold"] < test_fold]
        test = df[df["fold"] == test_fold].copy()

        best_score = -np.inf
        best_quantiles = None

        for long_q, short_q in product(long_grid, short_grid):

            fold_scores = []

            for validation_fold in train["fold"].unique()[1:]:

                history = train[train["fold"] < validation_fold]
                validation = train[train["fold"] == validation_fold]

                long_threshold = history["y_pred"].quantile(long_q)
                short_threshold = history["y_pred"].quantile(short_q)

                pos = np.select(
                    [
                        validation["y_pred"] >= long_threshold,
                        validation["y_pred"] <= short_threshold
                    ],
                    [1, -1],
                    default=0
                )

                pnl = (
                    pos * validation["y_true"]
                    - (pos != 0) * log_commission
                )

                if objective == "pnl":
                    score = pnl.sum()

                elif objective == "mean":
                    score = pnl.mean()

                elif objective == "sharpe":
                    std = pnl.std(ddof=1)
                    score = pnl.mean() / std if std > 0 else np.nan

                else:
                    raise ValueError(
                        "objective must be 'pnl', 'mean' or 'sharpe'"
                    )

                if np.isfinite(score):
                    fold_scores.append(score)

            if not fold_scores:
                continue

            score = np.mean(fold_scores)

            if score > best_score:
                best_score = score
                best_quantiles = long_q, short_q

        long_q, short_q = best_quantiles

        long_threshold = train["y_pred"].quantile(long_q)
        short_threshold = train["y_pred"].quantile(short_q)

        test["pos"] = np.select(
            [
                test["y_pred"] >= long_threshold,
                test["y_pred"] <= short_threshold
            ],
            [1, -1],
            default=0
        ).astype(int)

        result.append(test[["pos"]])

    return pd.concat(result).sort_index()

In [ ]:
positions = {
    model_name: walk_forward_positions(model_df, objective="pnl")
    for model_name, model_df in models.items()
}
positions.keys()

In [ ]:
models = {
    model_name: model_df.join(positions[model_name]).dropna(subset=["pos"])
    for model_name, model_df in models.items()
}

In [ ]:
for model_name, model_df in models.items():
    model_df.to_parquet(f"{model_name}_positions.parquet")

In [ ]:
model_names = list(models)
position_columns = pd.concat(
    [
        model_df[["pos"]].rename(columns={"pos": model_name})
        for model_name, model_df in models.items()
    ],
    axis=1,
)

dd = (
    df[["close"]]
    .join(position_columns, how="left")
    .dropna(subset=model_names)
)
dd[model_names] = dd[model_names].astype(int)

In [ ]:
dd.to_parquet('pos.parquet')

In [ ]:
model_names = list(models)
colors = sns.color_palette("tab20", n_colors=len(model_names)).as_hex()

price_return = dd["close"].shift(-1) / dd["close"] - 1
commission = 0.000

capital = {}

for model in model_names:
    strategy_return = (
        dd[model] * price_return
        - dd[model].abs() * commission
    )

    capital[model] = 100 * (1 + strategy_return.fillna(0)).cumprod()

fig = go.Figure()

for model, color in zip(model_names, colors):
    fig.add_trace(
        go.Scatter(
            x=dd.index,
            y=capital[model],
            name=model,
            line=dict(color=color, width=2)
        )
    )

fig.update_layout(
    template="plotly_white",
    title="",
    xaxis_title="",
    yaxis_title="Capital",
    hovermode="x unified",
    legend_title="")
fig.show()

In [ ]:
metrics = []
hours_per_year = 24 * 365
for model in model_names:
    ret = (
        dd[model] * price_return
        - dd[model].abs() * commission
    ).dropna()

    sharpe = np.sqrt(hours_per_year) * ret.mean() / ret.std()

    downside = ret[ret < 0]
    sortino = (
        np.sqrt(hours_per_year) * ret.mean() / downside.std()
        if len(downside) > 1 else np.nan
    )

    equity = 100 * (1 + ret).cumprod()

    drawdown = equity / equity.cummax() - 1
    max_dd = drawdown.min()

    metrics.append({
        "Model": model,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "MaxDD": max_dd
    })

metrics = pd.DataFrame(metrics).round(3)

metrics